# 🎲 Multiple Testing
**ISLP Chapter 13 · Pattern Recognition for the Rest of Us**

> Test 100 hypotheses at α=0.05 and you expect 5 false positives just by chance — even if nothing is real. Multiple testing procedures control this inflation of false discoveries.

### What you'll learn
- The multiple testing problem: why p-values get inflated
- Family-wise error rate (FWER) and Bonferroni correction
- False discovery rate (FDR) and the Benjamini-Hochberg procedure
- When to use FWER vs FDR
- Practical applications: genomics, A/B testing, feature selection

### Time: ~45 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
from statsmodels.stats.multitest import multipletests
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
print('✓ Setup complete')

## 🎯 Part 1 — The Problem: False Positives at Scale

At α=0.05, each test has a 5% chance of a false positive. If we run m independent tests, the probability of *at least one* false positive is:

```
P(at least one false positive) = 1 - (1-0.05)^m
```

For m=100 tests: 1 - 0.95^100 ≈ 99.4% chance of at least one false positive!

In [ ]:
# Simulate: false positive inflation
m_values = [1, 5, 10, 20, 50, 100, 200, 500, 1000]
alpha = 0.05
fwer = [1 - (1-alpha)**m for m in m_values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m_values, fwer, 'o-', color='#e85d20', lw=2.5, markersize=7)
ax.axhline(0.05, color='#888', lw=1, ls='--', label='Single test α=0.05')
ax.set_xlabel('Number of tests (m)')
ax.set_ylabel('Probability of at least one false positive')
ax.set_xscale('log')
ax.set_title('Multiple Testing Problem: FWER Inflation')
ax.legend()
plt.tight_layout(); plt.show()
print('📌 With 100 tests at α=0.05, there is a 99.4% chance of at least one false positive!')
print('   Bonferroni or BH correction is needed')

In [ ]:
# Concrete simulation
np.random.seed(42)
m = 100  # tests
n = 30   # observations per test

# All 100 null hypotheses are TRUE (no real effect)
# Any rejection is a false positive
p_values_null = [stats.ttest_ind(
    np.random.normal(0,1,n), np.random.normal(0,1,n)
).pvalue for _ in range(m)]

fig, ax = plt.subplots(figsize=(10, 3))
colors = ['#e85d20' if p < 0.05 else '#1e5fa8' for p in p_values_null]
ax.bar(range(m), sorted(p_values_null), color=[c for _,c in sorted(zip(p_values_null, colors))],
      edgecolor='white', width=0.9)
ax.axhline(0.05, color='black', lw=1.5, ls='--', label='α=0.05')
fp = sum(p < 0.05 for p in p_values_null)
ax.set_title(f'100 tests, ALL null true — {fp} false positives (orange) at α=0.05')
ax.set_xlabel('Test (sorted by p-value)'); ax.set_ylabel('p-value')
ax.legend()
plt.tight_layout(); plt.show()
print(f'False positives: {fp} out of {m} tests (expected: {m*0.05:.0f})')

In [ ]:
# Bonferroni vs BH correction
# Now: 80 true nulls, 20 real effects
np.random.seed(1)
m_null = 80; m_alt = 20; n = 30
p_null = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(0,1,n)).pvalue for _ in range(m_null)]
p_alt  = [stats.ttest_ind(np.random.normal(0,1,n), np.random.normal(2,1,n)).pvalue for _ in range(m_alt)]
p_all  = np.array(p_null + p_alt)
true_null = np.array([True]*m_null + [False]*m_alt)

results_corr = {}
for method, label in [('bonferroni','Bonferroni (FWER)'), ('fdr_bh','Benjamini-Hochberg (FDR)')]:
    reject, p_adj, _, _ = multipletests(p_all, alpha=0.05, method=method)
    tp = np.sum(reject & ~true_null)  # true positives
    fp = np.sum(reject & true_null)   # false positives
    fn = np.sum(~reject & ~true_null) # false negatives
    results_corr[label] = {'Rejected': reject.sum(), 'True Pos': tp, 'False Pos': fp, 'False Neg': fn, 'Power': tp/m_alt}
    print(f'{label}:')
    print(f'  Rejected: {reject.sum()} | TP={tp} | FP={fp} | FN={fn} | Power={tp/m_alt:.2f}')

print('\n📌 Bonferroni is very conservative — avoids false positives but misses many real effects')
print('   BH-FDR is more powerful — accepts some false positives to find more true effects')

In [ ]:
# Visualize: BH procedure step by step
sorted_p = np.sort(p_all)
m_total = len(p_all)
bh_threshold = np.arange(1, m_total+1) * 0.05 / m_total

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(1, m_total+1), sorted_p, s=20, alpha=0.6, label='p-values (sorted)',
          c=['#1a7a45' if p < 0.05 else '#888' for p in sorted(p_all)])
ax.plot(range(1, m_total+1), bh_threshold, color='#e85d20', lw=2, label='BH threshold k×α/m')
bonf_threshold = 0.05/m_total
ax.axhline(bonf_threshold, color='#c0392b', lw=1.5, ls='--', label=f'Bonferroni α/m={bonf_threshold:.4f}')
bh_reject_idx = np.where(sorted_p <= bh_threshold)[0]
if len(bh_reject_idx): ax.axvline(bh_reject_idx[-1]+1, color='#e85d20', lw=1, ls=':')
ax.set_xlabel('Rank (k)'); ax.set_ylabel('p-value')
ax.set_title('Benjamini-Hochberg Procedure: Reject all p ≤ k×α/m up to largest k')
ax.legend(); ax.set_ylim(0, 0.2)
plt.tight_layout(); plt.show()

In [ ]:
# Exercise workspace
# Task 1: Load a real genomics-style dataset and apply multiple testing correction
# Simulate 1000 gene expression comparisons with 50 truly differentially expressed genes
np.random.seed(42)
m_genes = 1000; m_de = 50
p_genes_null = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(0,1,20)).pvalue for _ in range(m_genes-m_de)]
p_genes_de   = [stats.ttest_ind(np.random.normal(0,1,20),np.random.normal(1.5,1,20)).pvalue for _ in range(m_de)]
p_genes_all  = np.array(p_genes_null + p_genes_de)
# YOUR CODE HERE: apply Bonferroni and BH. Compare TP, FP, power.

# Task 2: A/B testing scenario — you tested 20 website variants simultaneously
# p-values from each test are given below. Which pass Bonferroni? Which pass BH at q=0.05?
ab_pvalues = [0.001, 0.023, 0.041, 0.087, 0.12, 0.18, 0.002, 0.045, 0.22, 0.008,
              0.19, 0.31, 0.005, 0.11, 0.67, 0.043, 0.28, 0.0003, 0.55, 0.032]
# YOUR CODE HERE

In [ ]:
answers = {
    'q1': '',  # What is FWER and what does Bonferroni control?
    'q2': '',  # What is FDR and what does Benjamini-Hochberg control?
    'q3': '',  # When would you choose FDR correction over FWER correction?
    'q4': '',  # With 1000 tests at α=0.05, how many false positives do you expect without correction?
    'q5': '',  # What is the Bonferroni correction threshold for 50 tests at α=0.05?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

### 📤 Submit your results

In [ ]:
# ── Submit your results to the instructor ────────────────────────────────────
# Fill in your GitHub username (do this once, it stays saved in the notebook)
GITHUB_USERNAME = ""   # ← put your GitHub username here e.g. "jsmith42"

# ─────────────────────────────────────────────────────────────────────────────
# DO NOT EDIT BELOW THIS LINE
import json, urllib.parse

def build_submission_url(username, notebook_name, score, total, answers_dict,
                          form_base_url=None, entries=None):
    """Build a pre-filled Google Form URL for quiz submission."""
    if not form_base_url:
        print("⚠  No submission URL configured yet.")
        print("   Your instructor will share the Form URL — paste it into form_base_url below.")
        print("   Your answers are saved in this notebook regardless.")
        return None

    params = {
        entries.get('user',     'entry.000000001'): username,
        entries.get('notebook', 'entry.000000002'): notebook_name,
        entries.get('score',    'entry.000000003'): str(score),
        entries.get('total',    'entry.000000004'): str(total),
    }
    url = form_base_url.replace('/viewform','') + '/viewform?' + urllib.parse.urlencode(params)
    return url

# ── Submission config (your instructor fills these in) ───────────────────────
FORM_BASE_URL = None   # e.g. "https://docs.google.com/forms/d/e/XXXX/viewform"
FORM_ENTRIES  = {      # entry IDs from the Google Form
    'user':     'entry.000000001',
    'notebook': 'entry.000000002',
    'score':    'entry.000000003',
    'total':    'entry.000000004',
}
# ─────────────────────────────────────────────────────────────────────────────

# Calculate score from answers dict
score_val = sum(1 for v in answers.values() if str(v).strip())  # counts answered Qs
# For auto-graded notebooks the score is passed directly
# Here we just verify completion
n_answered = sum(1 for v in answers.values() if str(v).strip())
n_total    = len(answers)

print(f"\n{'='*50}")
print(f"Quiz completion: {n_answered}/{n_total} questions answered")
if n_answered < n_total:
    print(f"⚠  Please answer all {n_total} questions before submitting.")
else:
    print("✅ All questions answered!")
    if GITHUB_USERNAME:
        import os
        nb_name = os.path.basename(globals().get('__vsc_ipynb_file__',
                  globals().get('__file__', 'unknown-notebook'))).replace('.ipynb','')
        url = build_submission_url(GITHUB_USERNAME, nb_name,
                                   n_answered, n_total, answers,
                                   FORM_BASE_URL, FORM_ENTRIES)
        if url:
            print(f"\n🔗 Submit to instructor:")
            print(f"   {url}")
            print("\n   Copy the URL above, open it in a new tab, and click Submit.")
        else:
            print("\n   Save your notebook to GitHub to record your progress:")
            print("   File → Save a copy in GitHub → choose your fork")
    else:
        print("\n⚠  Add your GitHub username to GITHUB_USERNAME above, then re-run.")
    print(f"{'='*50}")

## 📚 Further Reading
- [ISLP Ch 13](https://intro-stat-learning.github.io/ISLP/) — Multiple Testing
- [statsmodels multipletests](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html)
- [Cross-Validation notebook](cross-validation.ipynb) — another way to control model selection error

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*